In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, random_split, ConcatDataset
from torch.cuda.amp import autocast, GradScaler
import time
import numpy as np

# --- CONFIGURATION ---
# Check for GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# --- 1. DATASET PREPARATION (The 70-10-20 Split) ---
def get_dataloaders(dataset_name='MNIST', batch_size=16):
    # ResNet requires some resizing/normalization.
    # We resize to 32x32 to be compatible with ResNet's downsampling
    transform = transforms.Compose([
        transforms.Resize((32, 32)),
        transforms.ToTensor(),
        transforms.Normalize((0.5,), (0.5,))
    ])

    # 1. Download/Load Data
    if dataset_name == 'MNIST':
        train_full = torchvision.datasets.MNIST(root='./data', train=True, download=True, transform=transform)
        test_full = torchvision.datasets.MNIST(root='./data', train=False, download=True, transform=transform)
    elif dataset_name == 'FashionMNIST':
        train_full = torchvision.datasets.FashionMNIST(root='./data', train=True, download=True, transform=transform)
        test_full = torchvision.datasets.FashionMNIST(root='./data', train=False, download=True, transform=transform)

    # 2. Merge and Re-split for strict 70-10-20 rule
    # Total MNIST/FashionMNIST size is 70,000
    full_dataset = ConcatDataset([train_full, test_full])
    total_size = len(full_dataset)

    train_size = int(0.70 * total_size)  # 49,000
    val_size = int(0.10 * total_size)    # 7,000
    test_size = total_size - train_size - val_size # 14,000 (20%)

    train_ds, val_ds, test_ds = random_split(full_dataset, [train_size, val_size, test_size])

    # 3. Create Loaders
    # pin_memory=False
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, pin_memory=False)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, pin_memory=False)
    test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False, pin_memory=False)

    return train_loader, val_loader, test_loader

# --- 2. MODEL SETUP (ResNet Modified for 1-Channel) ---
def get_model(model_name='resnet18'):
    if model_name == 'resnet18':
        model = torchvision.models.resnet18(pretrained=False)
    elif model_name == 'resnet50':
        model = torchvision.models.resnet50(pretrained=False)

    # MODIFY FIRST LAYER: Inputs are 1-channel (Grayscale), ResNet expects 3 (RGB)
    # We change input channels from 3 to 1.
    model.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)

    return model.to(device)

# --- 3. TRAINING LOOP (With AMP) ---
def train_and_evaluate(config):
    # Unpack config
    ds_name = config['dataset']
    model_name = config['model']
    batch_size = config['batch_size']
    opt_name = config['optimizer']
    lr = config['lr']
    epochs = config['epochs']

    print(f"\n--- Running: {ds_name} | {model_name} | {opt_name} | Batch {batch_size} | LR {lr} ---")

    # Get Data
    train_loader, val_loader, test_loader = get_dataloaders(ds_name, batch_size)

    # Get Model
    model = get_model(model_name)

    # Optimizer
    if opt_name == 'SGD':
        optimizer = optim.SGD(model.parameters(), lr=lr, momentum=0.9)
    elif opt_name == 'Adam':
        optimizer = optim.Adam(model.parameters(), lr=lr)

    criterion = nn.CrossEntropyLoss()
    scaler = GradScaler() # For Automatic Mixed Precision (AMP)

    # Training Loop
    best_val_acc = 0.0

    for epoch in range(epochs):
        model.train()
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()

            # AMP Forward Pass
            with autocast():
                outputs = model(images)
                loss = criterion(outputs, labels)

            # AMP Backward Pass
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

        # Validation Loop (Optional: Add print if you want to track progress)
        # We skip printing every epoch to save space, but you can add it back.

    # FINAL TEST EVALUATION
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    acc = 100 * correct / total
    print(f"Result >> Test Accuracy: {acc:.2f}%")
    return acc

# --- 4. EXPERIMENT RUNNER ---
# This matches the table in your assignment image
# You can add FashionMNIST to this list to do Q1(a) fully.
# --- 4. EXPERIMENT RUNNER (MODIFIED FOR BOTH DATASETS) ---
experiments = [
    # Format: (Batch, Optim, LR, Model)
    (16, 'SGD', 0.001, 'resnet18'),
    (16, 'SGD', 0.0001, 'resnet18'),
    (16, 'Adam', 0.001, 'resnet18'),
    (16, 'Adam', 0.0001, 'resnet18'),

    (16, 'SGD', 0.001, 'resnet50'),
    (16, 'SGD', 0.0001, 'resnet50'),
    (16, 'Adam', 0.001, 'resnet50'),
    (16, 'Adam', 0.0001, 'resnet50'),

    (32, 'SGD', 0.001, 'resnet18'),
    (32, 'Adam', 0.0001, 'resnet18'),

    (32, 'SGD', 0.001, 'resnet50'),
    (32, 'Adam', 0.0001, 'resnet50'),

    # NOTE: You can add the Batch 32 experiments here if you need them later
]

datasets_to_run = ['MNIST', 'FashionMNIST']

print(f"Total Configurations to run: {len(datasets_to_run) * len(experiments)}")

for ds_name in datasets_to_run:
    print(f"\n{'='*20} STARTING {ds_name} EXPERIMENTS {'='*20}")

    for exp in experiments:
        batch, opt, lr, model = exp
        config = {
            'dataset': ds_name,   # Loops through MNIST then FashionMNIST
            'batch_size': batch,
            'optimizer': opt,
            'lr': lr,
            'model': model,
            'epochs': 5  # Keep this low (e.g. 5) to finish before the deadline!
        }

        # Run the training
        train_and_evaluate(config)

print("\nALL EXPERIMENTS COMPLETED.")

Using device: cuda
Total Configurations to run: 24

==================== STARTING MNIST EXPERIMENTS ====================

--- Running: MNIST | resnet18 | SGD | Batch 16 | LR 0.001 ---


100%|██████████| 9.91M/9.91M [00:01<00:00, 4.98MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 131kB/s]
100%|██████████| 1.65M/1.65M [00:01<00:00, 1.22MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 8.82MB/s]
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)
/tmp/ipython-input-3246436826.py:91: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler() # For Automatic Mixed Precision (AMP)
/tmp/ipython-input-3246436826.py:104: FutureWarning: `

Result >> Test Accuracy: 98.96%

--- Running: MNIST | resnet18 | SGD | Batch 16 | LR 0.0001 ---
Result >> Test Accuracy: 98.43%

--- Running: MNIST | resnet18 | Adam | Batch 16 | LR 0.001 ---
Result >> Test Accuracy: 98.81%

--- Running: MNIST | resnet18 | Adam | Batch 16 | LR 0.0001 ---
Result >> Test Accuracy: 98.49%

--- Running: MNIST | resnet50 | SGD | Batch 16 | LR 0.001 ---
Result >> Test Accuracy: 98.57%

--- Running: MNIST | resnet50 | SGD | Batch 16 | LR 0.0001 ---
Result >> Test Accuracy: 97.94%

--- Running: MNIST | resnet50 | Adam | Batch 16 | LR 0.001 ---
Result >> Test Accuracy: 98.40%

--- Running: MNIST | resnet50 | Adam | Batch 16 | LR 0.0001 ---
Result >> Test Accuracy: 98.33%

--- Running: MNIST | resnet18 | SGD | Batch 32 | LR 0.001 ---
Result >> Test Accuracy: 98.95%

--- Running: MNIST | resnet18 | Adam | Batch 32 | LR 0.0001 ---
Result >> Test Accuracy: 98.49%

--- Running: MNIST | resnet50 | SGD | Batch 32 | LR 0.001 ---
Result >> Test Accuracy: 98.60%

--- Run

100%|██████████| 26.4M/26.4M [00:02<00:00, 10.4MB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 189kB/s]
100%|██████████| 4.42M/4.42M [00:01<00:00, 3.31MB/s]
100%|██████████| 5.15k/5.15k [00:00<00:00, 14.4MB/s]


Result >> Test Accuracy: 90.07%

--- Running: FashionMNIST | resnet18 | SGD | Batch 16 | LR 0.0001 ---
Result >> Test Accuracy: 89.05%

--- Running: FashionMNIST | resnet18 | Adam | Batch 16 | LR 0.001 ---
Result >> Test Accuracy: 90.68%

--- Running: FashionMNIST | resnet18 | Adam | Batch 16 | LR 0.0001 ---
Result >> Test Accuracy: 90.20%

--- Running: FashionMNIST | resnet50 | SGD | Batch 16 | LR 0.001 ---
Result >> Test Accuracy: 89.01%

--- Running: FashionMNIST | resnet50 | SGD | Batch 16 | LR 0.0001 ---
Result >> Test Accuracy: 84.59%

--- Running: FashionMNIST | resnet50 | Adam | Batch 16 | LR 0.001 ---
Result >> Test Accuracy: 85.12%

--- Running: FashionMNIST | resnet50 | Adam | Batch 16 | LR 0.0001 ---
Result >> Test Accuracy: 88.10%

--- Running: FashionMNIST | resnet18 | SGD | Batch 32 | LR 0.001 ---
Result >> Test Accuracy: 90.40%

--- Running: FashionMNIST | resnet18 | Adam | Batch 32 | LR 0.0001 ---
Result >> Test Accuracy: 89.84%

--- Running: FashionMNIST | resnet50 | 

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, random_split, ConcatDataset
from torch.cuda.amp import autocast, GradScaler
import time
import numpy as np

# --- CONFIGURATION ---
# Check for GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# --- 1. DATASET PREPARATION (The 70-10-20 Split) ---
def get_dataloaders(dataset_name='MNIST', batch_size=16):
    # ResNet requires some resizing/normalization.
    # We resize to 32x32 to be compatible with ResNet's downsampling
    transform = transforms.Compose([
        transforms.Resize((32, 32)),
        transforms.ToTensor(),
        transforms.Normalize((0.5,), (0.5,))
    ])

    # 1. Download/Load Data
    if dataset_name == 'MNIST':
        train_full = torchvision.datasets.MNIST(root='./data', train=True, download=True, transform=transform)
        test_full = torchvision.datasets.MNIST(root='./data', train=False, download=True, transform=transform)
    elif dataset_name == 'FashionMNIST':
        train_full = torchvision.datasets.FashionMNIST(root='./data', train=True, download=True, transform=transform)
        test_full = torchvision.datasets.FashionMNIST(root='./data', train=False, download=True, transform=transform)

    # 2. Merge and Re-split for strict 70-10-20 rule
    # Total MNIST/FashionMNIST size is 70,000
    full_dataset = ConcatDataset([train_full, test_full])
    total_size = len(full_dataset)

    train_size = int(0.70 * total_size)  # 49,000
    val_size = int(0.10 * total_size)    # 7,000
    test_size = total_size - train_size - val_size # 14,000 (20%)

    train_ds, val_ds, test_ds = random_split(full_dataset, [train_size, val_size, test_size])

    # 3. Create Loaders
    # pin_memory=False
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, pin_memory=False)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, pin_memory=False)
    test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False, pin_memory=False)

    return train_loader, val_loader, test_loader

# --- 2. MODEL SETUP (ResNet Modified for 1-Channel) ---
def get_model(model_name='resnet18'):
    if model_name == 'resnet18':
        model = torchvision.models.resnet18(pretrained=False)
    elif model_name == 'resnet50':
        model = torchvision.models.resnet50(pretrained=False)

    # MODIFY FIRST LAYER: Inputs are 1-channel (Grayscale), ResNet expects 3 (RGB)
    # We change input channels from 3 to 1.
    model.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)

    return model.to(device)

# --- 3. TRAINING LOOP (With AMP) ---
def train_and_evaluate(config):
    # Unpack config
    ds_name = config['dataset']
    model_name = config['model']
    batch_size = config['batch_size']
    opt_name = config['optimizer']
    lr = config['lr']
    epochs = config['epochs']

    print(f"\n--- Running: {ds_name} | {model_name} | {opt_name} | Batch {batch_size} | LR {lr} ---")

    # Get Data
    train_loader, val_loader, test_loader = get_dataloaders(ds_name, batch_size)

    # Get Model
    model = get_model(model_name)

    # Optimizer
    if opt_name == 'SGD':
        optimizer = optim.SGD(model.parameters(), lr=lr, momentum=0.9)
    elif opt_name == 'Adam':
        optimizer = optim.Adam(model.parameters(), lr=lr)

    criterion = nn.CrossEntropyLoss()
    scaler = GradScaler() # For Automatic Mixed Precision (AMP)

    # Training Loop
    best_val_acc = 0.0

    for epoch in range(epochs):
        model.train()
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()

            # AMP Forward Pass
            with autocast():
                outputs = model(images)
                loss = criterion(outputs, labels)

            # AMP Backward Pass
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

        # Validation Loop (Optional: Add print if you want to track progress)
        # We skip printing every epoch to save space, but you can add it back.

    # FINAL TEST EVALUATION
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    acc = 100 * correct / total
    print(f"Result >> Test Accuracy: {acc:.2f}%")
    return acc

# --- 4. EXPERIMENT RUNNER ---
# This matches the table in your assignment image
# You can add FashionMNIST to this list to do Q1(a) fully.
# --- 4. EXPERIMENT RUNNER (MODIFIED FOR BOTH DATASETS) ---
experiments = [
    # Format: (Batch, Optim, LR, Model)
    (16, 'SGD', 0.001, 'resnet18'),
    (16, 'SGD', 0.0001, 'resnet18'),
    (16, 'Adam', 0.001, 'resnet18'),
    (16, 'Adam', 0.0001, 'resnet18'),

    (16, 'SGD', 0.001, 'resnet50'),
    (16, 'SGD', 0.0001, 'resnet50'),
    (16, 'Adam', 0.001, 'resnet50'),
    (16, 'Adam', 0.0001, 'resnet50'),

    (32, 'SGD', 0.001, 'resnet18'),
    (32, 'Adam', 0.0001, 'resnet18'),

    (32, 'SGD', 0.001, 'resnet50'),
    (32, 'Adam', 0.0001, 'resnet50'),

    # NOTE: You can add the Batch 32 experiments here if you need them later
]

datasets_to_run = ['MNIST', 'FashionMNIST']

print(f"Total Configurations to run: {len(datasets_to_run) * len(experiments)}")

for ds_name in datasets_to_run:
    print(f"\n{'='*20} STARTING {ds_name} EXPERIMENTS {'='*20}")

    for exp in experiments:
        batch, opt, lr, model = exp
        config = {
            'dataset': ds_name,   # Loops through MNIST then FashionMNIST
            'batch_size': batch,
            'optimizer': opt,
            'lr': lr,
            'model': model,
            'epochs': 2  # Keep this low (e.g. 5) to finish before the deadline!
        }

        # Run the training
        train_and_evaluate(config)

print("\nALL EXPERIMENTS COMPLETED.")

Using device: cuda
Total Configurations to run: 24

==================== STARTING MNIST EXPERIMENTS ====================

--- Running: MNIST | resnet18 | SGD | Batch 16 | LR 0.001 ---


/tmp/ipython-input-6449677.py:91: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler() # For Automatic Mixed Precision (AMP)
/tmp/ipython-input-6449677.py:104: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


Result >> Test Accuracy: 98.92%

--- Running: MNIST | resnet18 | SGD | Batch 16 | LR 0.0001 ---
Result >> Test Accuracy: 97.81%

--- Running: MNIST | resnet18 | Adam | Batch 16 | LR 0.001 ---
Result >> Test Accuracy: 98.51%

--- Running: MNIST | resnet18 | Adam | Batch 16 | LR 0.0001 ---
Result >> Test Accuracy: 98.51%

--- Running: MNIST | resnet50 | SGD | Batch 16 | LR 0.001 ---
Result >> Test Accuracy: 98.18%

--- Running: MNIST | resnet50 | SGD | Batch 16 | LR 0.0001 ---
Result >> Test Accuracy: 95.91%

--- Running: MNIST | resnet50 | Adam | Batch 16 | LR 0.001 ---
Result >> Test Accuracy: 96.92%

--- Running: MNIST | resnet50 | Adam | Batch 16 | LR 0.0001 ---
Result >> Test Accuracy: 96.19%

--- Running: MNIST | resnet18 | SGD | Batch 32 | LR 0.001 ---
Result >> Test Accuracy: 98.69%

--- Running: MNIST | resnet18 | Adam | Batch 32 | LR 0.0001 ---
Result >> Test Accuracy: 98.45%

--- Running: MNIST | resnet50 | SGD | Batch 32 | LR 0.001 ---
Result >> Test Accuracy: 97.64%

--- Run

In [ ]:
import time
import numpy as np
from sklearn import svm
from sklearn.metrics import accuracy_score
import torchvision
import torchvision.transforms as transforms

# --- CONFIGURATION ---
# To finish before the deadline, we limit training data to 10k samples.
# Set to None if you want to run on the full 60k dataset (Takes Hours!)
LIMIT_SAMPLES = 10000

def load_and_prep_data(dataset_name):
    print(f"Loading {dataset_name}...")

    # 1. Load Data (Reuse torchvision to get the same data)
    if dataset_name == 'MNIST':
        train_set = torchvision.datasets.MNIST(root='./data', train=True, download=True)
        test_set = torchvision.datasets.MNIST(root='./data', train=False, download=True)
    else:
        train_set = torchvision.datasets.FashionMNIST(root='./data', train=True, download=True)
        test_set = torchvision.datasets.FashionMNIST(root='./data', train=False, download=True)

    # 2. Convert to Numpy & Flatten (28x28 -> 784 features)
    # SVMs expect a 1D array of features per image
    X_train = train_set.data.numpy().reshape(-1, 28*28)
    y_train = train_set.targets.numpy()

    X_test = test_set.data.numpy().reshape(-1, 28*28)
    y_test = test_set.targets.numpy()

    # 3. Normalize (0-255 -> 0-1)
    # Crucial for SVM convergence!
    X_train = X_train / 255.0
    X_test = X_test / 255.0

    # 4. Limit Samples (Optional, for speed)
    if LIMIT_SAMPLES:
        print(f"  [Info] Subsampling dataset to {LIMIT_SAMPLES} samples for speed...")
        X_train = X_train[:LIMIT_SAMPLES]
        y_train = y_train[:LIMIT_SAMPLES]
        # We can keep the full test set or reduce it too, usually full test set is fast enough to predict

    return X_train, y_train, X_test, y_test

def run_svm_experiment(dataset_name, kernel_type):
    # Load Data
    X_train, y_train, X_test, y_test = load_and_prep_data(dataset_name)

    print(f"Training SVM ({dataset_name} | Kernel: {kernel_type})...")

    # Initialize Model
    # gamma='scale' is standard for RBF/Poly
    clf = svm.SVC(kernel=kernel_type, gamma='scale')

    # Train & Time
    start_time = time.time()
    clf.fit(X_train, y_train)
    end_time = time.time()

    training_time_ms = (end_time - start_time) * 1000

    # Test
    print("Evaluating...")
    y_pred = clf.predict(X_test)
    acc = accuracy_score(y_test, y_pred) * 100

    print(f"RESULT: Dataset: {dataset_name} | Kernel: {kernel_type}")
    print(f"  > Accuracy: {acc:.2f}%")
    print(f"  > Training Time: {training_time_ms:.0f} ms")
    print("-" * 40)

    return {
        'dataset': dataset_name,
        'kernel': kernel_type,
        'accuracy': acc,
        'time_ms': training_time_ms
    }

# --- RUN EXPERIMENTS ---
datasets = ['MNIST', 'FashionMNIST']
kernels = ['poly', 'rbf']

results = []

for ds in datasets:
    for k in kernels:
        res = run_svm_experiment(ds, k)
        results.append(res)

print("\n=== FINAL SUMMARY TABLE FOR Q1(b) ===")
print(f"{'Dataset':<15} | {'Kernel':<10} | {'Accuracy':<10} | {'Time (ms)':<10}")
print("-" * 55)
for r in results:
    print(f"{r['dataset']:<15} | {r['kernel']:<10} | {r['accuracy']:.2f}%     | {r['time_ms']:.0f}")

Loading MNIST...
  [Info] Subsampling dataset to 10000 samples for speed...
Training SVM (MNIST | Kernel: poly)...
Evaluating...
RESULT: Dataset: MNIST | Kernel: poly
  > Accuracy: 95.15%
  > Training Time: 8372 ms
----------------------------------------
Loading MNIST...
  [Info] Subsampling dataset to 10000 samples for speed...
Training SVM (MNIST | Kernel: rbf)...
Evaluating...
RESULT: Dataset: MNIST | Kernel: rbf
  > Accuracy: 95.94%
  > Training Time: 8789 ms
----------------------------------------
Loading FashionMNIST...
  [Info] Subsampling dataset to 10000 samples for speed...
Training SVM (FashionMNIST | Kernel: poly)...
Evaluating...
RESULT: Dataset: FashionMNIST | Kernel: poly
  > Accuracy: 81.66%
  > Training Time: 8664 ms
----------------------------------------
Loading FashionMNIST...
  [Info] Subsampling dataset to 10000 samples for speed...
Training SVM (FashionMNIST | Kernel: rbf)...
Evaluating...
RESULT: Dataset: FashionMNIST | Kernel: rbf
  > Accuracy: 85.31%
  > Tr